# Bản sao 292p của **tập val** (`ek100-292-val-0`, `ek100-292-val-1`), chỉ cần CPU

Tạo lại 2 dataset val 292p mà các notebook bước 2 (`kaggle_extract_pooled`) và nghiên cứu predictor cần. Cách làm giống
`kaggle_reencode_train_hf`: resize đúng như eval (cv2 `INTER_LINEAR`, cạnh ngắn 292), giữ mọi frame và đúng fps, kiểm tra số
frame/fps với bản gốc (`manifest.csv`).

- **Nén giống bản train** (`yuv420p` crf 18), để train và val khác nhau đúng ở nội dung, không ở mức nén.
  (Bản val cũ của repo dùng `yuv444p` crf 12; muốn y hệt bản cũ thì đổi `CRF`, `PIX_FMT` ở cell dưới.)
- **Nguồn: chỉ có Bristol.** 138 video val đều là video EK55 (P01_11 …), bản sao Hugging Face không có. Cell 3 thử tải 1 MB từ
  data.bris.ac.uk trước; nếu server còn lỗi, notebook dừng ngay (không tốn thời gian), thử lại sau.

**Chạy 2 phiên:** `SHARD_ID = 0` và `SHARD_ID = 1` (song song được). Mỗi phiên khoảng 6.6 giờ video, tức khoảng 5–6 giờ encode
(0.6x realtime mỗi luồng, 2 luồng) cộng thời gian tải từ Bristol (khoảng 90 GB mỗi shard, tốc độ Bristol trước đây 4–10 MB/s, nên
có thể là phần chậm nhất). Hết giờ thì *Add Input* output của phiên đó rồi chạy lại cùng `SHARD_ID`.

Xong mỗi phiên: tab **Output → New Dataset**, đặt tên **`ek100-292-val-0`** (shard 0) và **`ek100-292-val-1`** (shard 1).

**Cài đặt:** Accelerator **None**, Internet **On**.

In [ ]:
SHARD_ID = 0        # 0 hoặc 1
NUM_SHARDS = 2
CRF, PIX_FMT = 18, "yuv420p"   # giống bản train; bản val cũ của repo: 12, "yuv444p"

In [ ]:
BRANCH = "direction2-balanced-head"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /tmp/repo
!apt-get -qq install -y aria2 ffmpeg > /dev/null 2>&1; pip install -q decord av imageio-ffmpeg
R = "/tmp/repo"
!git -C $R log --oneline -1
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /tmp/ek100-ann

In [ ]:
# Is data.bris.ac.uk reachable? Download 1 MB of the first val videos of this shard; stop at once if not.
import subprocess, sys, pandas as pd
sys.path.insert(0, f"{R}/scripts")
from stream_eval_ek100 import url_candidates
vids = sorted(pd.read_csv("/tmp/ek100-ann/EPIC_100_validation.csv").video_id.unique())[SHARD_ID::NUM_SHARDS]
print(f"shard {SHARD_ID}: {len(vids)} val videos")
ok = 0
for v in vids[:3]:
    for u in url_candidates(v):
        r = subprocess.run(["curl", "-sS", "-k", "-L", "-A", "Mozilla/5.0", "-r", "0-1048575", "-o", "/dev/null",
                            "-w", "%{http_code} %{size_download}", "--max-time", "60", u], capture_output=True, text=True)
        code_, size = (r.stdout.split() + ["0", "0"])[:2]
        print(f"  {v}: HTTP {code_}, {size} bytes  {u.split('/datasets/')[-1]}  {r.stderr.strip()[-120:]}")
        if code_ in ("200", "206") and float(size) > 0:
            ok += 1
            break
if ok == 0:
    raise SystemExit("data.bris.ac.uk is still not serving the videos (connection reset / timeout). Try again later.")
print("Bristol reachable: starting.")

In [ ]:
# Resume: copy the output of an earlier run of THIS shard (attach it via Add Input) so the new output is complete
import glob, os, shutil
OUT = "/kaggle/working/ek100_292"
prev = [os.path.dirname(m) for m in glob.glob("/kaggle/input/**/ek100_292/manifest.csv", recursive=True)]
for d in prev:
    print("copying", d)
    shutil.copytree(d, OUT, dirs_exist_ok=True)
print("resumed from", len(prev), "earlier output(s)" if prev else "(fresh run)")

In [ ]:
!python $R/scripts/reencode_ek100.py --split val --source bristol --shard_id $SHARD_ID --num_shards $NUM_SHARDS \
    --crf $CRF --pix_fmt $PIX_FMT --annotations_dir /tmp/ek100-ann --out_dir /kaggle/working/ek100_292 \
    --tmp_dir /kaggle/tmp/raw --encode_workers 2 --ffmpeg_threads 2 --parallel_downloads 3 --max_hours 11 --max_output_gb 19

In [ ]:
import pandas as pd
m = pd.read_csv("/kaggle/working/ek100_292/manifest.csv", dtype=str)
ok = m.ok == "True"
print(f"{ok.sum()} ok, {(~ok).sum()} failed (shard {SHARD_ID} has {len(vids)} videos)")
print("median bitrate (Mbit/s):", pd.to_numeric(m.mbps[ok], errors="coerce").median())
filled = pd.to_numeric(m.filled_frames, errors="coerce").fillna(0)
print("videos with frames filled in (corrupt packets in the original):\n", m.loc[filled > 0, ["video_id", "filled_frames"]])
print(m.loc[~ok, ["video_id", "error"]])
!du -sh /kaggle/working/ek100_292